# Multi-Table Agent Test Notebook

This notebook tests the multi-table query support introduced in the `multitable` branch:
- `ColumnSchema`, `TableSchema`, `DatabaseSchema` dataclasses
- `get_compact_summary()` and `get_full_schema_str()` output
- Backward compatibility (no schema → single-table legacy mode)
- Two-table agent run with JOIN query
- Table-selection path (forced via low `compact_threshold`)
- API schema dict parsing via `_schema_from_dict`

## 1. Setup and Imports

**Purpose:** Import all modules including the three new schema classes. A successful import confirms `Agent/schema.py` is accessible and exports are wired correctly through `Agent/__init__.py`.

In [ ]:
import os
import sys
import shutil
import tempfile
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath('../test_code')))

from src.arco import SalesDataWorkflow, DatabaseSchema, TableSchema, ColumnSchema
from agent_api import _schema_from_dict

OPENAI_AVAILABLE = bool(os.getenv("OPENAI_API_KEY"))
print("All imports successful!")
print(f"OpenAI API key available: {OPENAI_AVAILABLE}")

## 2. Schema Class Unit Tests (no LLM required)

**Purpose:** Verify `ColumnSchema` stores all fields correctly including optional ones, and that `nullable` defaults to `True`. These fields are the building blocks that flow into the LLM prompts.

In [ ]:
# --- ColumnSchema ---
col_minimal = ColumnSchema(name="Store_Number", description="Unique store identifier")
col_full = ColumnSchema(
    name="Sold_Date",
    description="Transaction date",
    data_type="DATE",
    example_values=["2022-01-01", "2023-06-15"],
    nullable=False,
)

print("ColumnSchema (minimal):")
print(f"  name={col_minimal.name}, description={col_minimal.description}")
print(f"  data_type={col_minimal.data_type} (default)")  # VARCHAR
print(f"  nullable={col_minimal.nullable} (default)")     # True

print("\nColumnSchema (full):")
print(f"  name={col_full.name}, data_type={col_full.data_type}")
print(f"  example_values={col_full.example_values}")
print(f"  nullable={col_full.nullable}")

assert col_minimal.data_type == "VARCHAR"
assert col_minimal.nullable is True
assert col_full.nullable is False
print("\nColumnSchema assertions passed!")

**Purpose:** Build a two-table `DatabaseSchema` and verify `get_compact_summary()` produces the short listing (names + descriptions only) and `get_full_schema_str()` produces the detailed listing with column metadata. The compact form is what the LLM sees in the table-selection step; the full form is what it sees for SQL generation.

In [ ]:
# Build a two-table schema
schema = DatabaseSchema(
    tables=[
        TableSchema(
            name="sales",
            description="Daily store-level sales transactions with quantity and revenue",
            file_path="/placeholder/sales.parquet",
            columns=[
                ColumnSchema("Store_Number", "Unique store identifier", "INTEGER",
                             example_values=["1", "2", "3"]),
                ColumnSchema("SKU_Coded", "Product SKU code (joins with products)", "INTEGER"),
                ColumnSchema("Sold_Date", "Transaction date", "DATE", nullable=False),
                ColumnSchema("Qty_Sold", "Number of units sold", "FLOAT"),
                ColumnSchema("Total_Sale_Value", "Total revenue in USD", "FLOAT"),
                ColumnSchema("On_Promo", "1 if item was on promotion, 0 otherwise", "INTEGER",
                             example_values=["0", "1"]),
            ],
        ),
        TableSchema(
            name="products",
            description="Product catalog mapping SKU codes to product classes",
            file_path="/placeholder/products.parquet",
            columns=[
                ColumnSchema("SKU_Coded", "Product SKU code (joins with sales)", "INTEGER", nullable=False),
                ColumnSchema("Product_Class_Code", "Numeric product class identifier", "INTEGER"),
            ],
        ),
    ],
    compact_threshold=5,
)

print(f"Tables registered: {[t.name for t in schema.tables]}")
print(f"compact_threshold: {schema.compact_threshold}")
print(f"should_use_table_selection (2 tables, threshold=5): {schema.should_use_table_selection()}")
assert schema.should_use_table_selection() is False

print("\nSchema assertions passed!")

In [ ]:
# Inspect compact summary
compact = schema.get_compact_summary()
print("=== Compact Summary (used in table-selection prompt) ===")
print(compact)

assert "Table: sales" in compact
assert "Table: products" in compact
assert "SKU_Coded" not in compact, "Compact summary should NOT include column details"
print("\nCompact summary assertions passed!")

In [ ]:
# Inspect full schema string (all tables)
full = schema.get_full_schema_str()
print("=== Full Schema String (used in SQL generation prompt) ===")
print(full)

assert "SKU_Coded" in full
assert "NOT NULL" in full           # Sold_Date and SKU_Coded in products are non-nullable
assert "examples: 1, 2, 3" in full  # example_values for Store_Number
print("\nFull schema string assertions passed!")

In [ ]:
# Inspect full schema string filtered to one table
sales_only = schema.get_full_schema_str(table_names=["sales"])
print("=== Full Schema String (sales table only) ===")
print(sales_only)

assert "Table: sales" in sales_only
assert "Table: products" not in sales_only   # section header absent, even if descriptions mention "products"
assert "Product_Class_Code" not in sales_only
print("\nFiltered schema string assertions passed!")

In [ ]:
# Test get_table() lookup
sales_table = schema.get_table("sales")
missing_table = schema.get_table("does_not_exist")

assert sales_table is not None
assert sales_table.name == "sales"
assert len(sales_table.columns) == 6
assert missing_table is None
print(f"get_table('sales') returned: {sales_table.name} with {len(sales_table.columns)} columns")
print(f"get_table('does_not_exist') returned: {missing_table}")
print("\nget_table assertions passed!")

In [ ]:
# Test should_use_table_selection with low threshold
schema_low_threshold = DatabaseSchema(
    tables=schema.tables,
    compact_threshold=1,  # Force table selection even with 2 tables
)
assert schema_low_threshold.should_use_table_selection() is True
print(f"With compact_threshold=1 and 2 tables: should_use_table_selection = {schema_low_threshold.should_use_table_selection()}")
print("\ncompact_threshold assertions passed!")

## 3. Prepare Test Parquet Files

**Purpose:** Split the single existing parquet file into two tables — `sales` (transactions) and `products` (SKU → Product_Class_Code mapping) — to simulate a real multi-table schema. This lets us test JOIN queries like "show revenue by product class" without requiring extra data files.

In [ ]:
# Load the original single-table dataset using the same path the agent uses
from core.state import DEFAULT_DATA_PATH

ORIGINAL_PATH = DEFAULT_DATA_PATH
df_original = pd.read_parquet(ORIGINAL_PATH)
print(f"Loaded from: {os.path.basename(ORIGINAL_PATH)}")
print(f"Original dataset: {df_original.shape[0]:,} rows, columns: {list(df_original.columns)}")
print(df_original.head(3))

In [ ]:
# Create a temp directory for the split parquet files
TEST_DATA_DIR = tempfile.mkdtemp(prefix="multitable_test_data_")
SALES_PATH = os.path.join(TEST_DATA_DIR, "sales.parquet")
PRODUCTS_PATH = os.path.join(TEST_DATA_DIR, "products.parquet")

# sales: all transaction columns except Product_Class_Code (moved to products)
df_sales = df_original[["Store_Number", "SKU_Coded", "Sold_Date", "Qty_Sold", "Total_Sale_Value", "On_Promo"]].copy()

# products: unique SKU → Product_Class_Code mapping
df_products = (
    df_original[["SKU_Coded", "Product_Class_Code"]]
    .drop_duplicates(subset=["SKU_Coded"])
    .reset_index(drop=True)
)

df_sales.to_parquet(SALES_PATH, index=False)
df_products.to_parquet(PRODUCTS_PATH, index=False)

print(f"sales.parquet: {df_sales.shape[0]:,} rows, columns: {list(df_sales.columns)}")
print(f"products.parquet: {df_products.shape[0]:,} rows, columns: {list(df_products.columns)}")
print(f"\nFiles written to: {TEST_DATA_DIR}")

## 4. Backward Compatibility — Single-Table Legacy Mode

**Purpose:** Confirm that `SalesDataAgent()` with no `schema` argument still works exactly as before. The agent should auto-build a minimal `DatabaseSchema` from `DEFAULT_DATA_PATH` at query time, so the user sees no difference from the old API.

In [ ]:
# Verify agent accepts no schema (legacy mode)
try:
    legacy_agent = SalesDataWorkflow(model="gpt-4o-mini")
    assert legacy_agent.schema is None, "schema should be None when not provided"
    print(f"Legacy agent initialized: schema={legacy_agent.schema} (None = backward compat OK)")
    print(f"  data_path: {os.path.basename(legacy_agent.data_path)}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
if OPENAI_AVAILABLE:
    print("Running legacy single-table query (backward compat check)...")
    result, _ = legacy_agent.run_script(
        "What is the total revenue in November 2022?",
        no_vis=True,
    )
    print(f"\nSQL: {result.get('sql_query', '').strip()}")
    print(f"Answer: {result.get('answer', [''])[0][:120]}")
    assert "error" not in result or not result["error"], f"Got error: {result.get('error')}"
    print("\nBackward compat test passed!")
else:
    print("Skipping live query (OPENAI_API_KEY not set)")

## 5. Define a Two-Table DatabaseSchema

**Purpose:** Build the `DatabaseSchema` that describes the two parquet files created above. Each column has a meaningful description and the correct `data_type`. This is the standard the LLM uses to understand what data is available and how the tables relate to each other.

In [ ]:
multitable_schema = DatabaseSchema(
    tables=[
        TableSchema(
            name="sales",
            description="Daily store-level sales transactions with quantity and revenue figures",
            file_path=SALES_PATH,
            columns=[
                ColumnSchema("Store_Number", "Unique store identifier", "INTEGER",
                             example_values=["1", "2", "3"]),
                ColumnSchema("SKU_Coded", "Product SKU code — foreign key to products.SKU_Coded", "INTEGER"),
                ColumnSchema("Sold_Date", "Transaction date (YYYY-MM-DD)", "DATE", nullable=False),
                ColumnSchema("Qty_Sold", "Number of units sold in this transaction", "FLOAT"),
                ColumnSchema("Total_Sale_Value", "Total revenue in USD for this transaction", "FLOAT"),
                ColumnSchema("On_Promo", "Promotional flag: 1 = on promotion, 0 = regular price", "INTEGER",
                             example_values=["0", "1"]),
            ],
        ),
        TableSchema(
            name="products",
            description="Product catalog mapping SKU codes to product class codes",
            file_path=PRODUCTS_PATH,
            columns=[
                ColumnSchema("SKU_Coded", "Unique product SKU code — primary key, joins with sales.SKU_Coded",
                             "INTEGER", nullable=False),
                ColumnSchema("Product_Class_Code", "Numeric code identifying the product class/category",
                             "INTEGER"),
            ],
        ),
    ],
    compact_threshold=5,  # 2 tables < 5 → full schema goes directly to SQL prompt
)

print("DatabaseSchema created:")
print(f"  Tables: {[t.name for t in multitable_schema.tables]}")
print(f"  compact_threshold: {multitable_schema.compact_threshold}")
print(f"  should_use_table_selection: {multitable_schema.should_use_table_selection()}")
print("\n=== Full schema that will be sent to SQL generation ===")
print(multitable_schema.get_full_schema_str())

## 6. Two-Table Agent Run (Requires OpenAI)

**Purpose:** Instantiate `SalesDataAgent` with the two-table schema and run queries that require JOINs. Since `compact_threshold=5` and there are only 2 tables, the full schema is passed directly to the SQL generation prompt — no table-selection step is needed.

In [ ]:
agent = None
if OPENAI_AVAILABLE:
    agent = SalesDataWorkflow(
        model="gpt-4o-mini",
        schema=multitable_schema,
    )
    assert agent.schema is multitable_schema
    print(f"Multi-table agent initialized with schema: {[t.name for t in agent.schema.tables]}")
else:
    print("Skipping agent initialization (OPENAI_API_KEY not set)")

In [ ]:
# Test 1: single-table query (should use only the sales table)
if OPENAI_AVAILABLE:
    print("--- Test: single-table query ---")
    result_single, _ = agent.run(
        "What is the total revenue in November 2022?",
        no_vis=True,
    )
    print(f"SQL: {result_single.get('sql_query', '').strip()}")
    print(f"Answer: {result_single.get('answer', [''])[0][:150]}")
    assert "error" not in result_single or not result_single["error"]
    print("\nSingle-table query on multi-table schema: PASSED")
else:
    print("Skipping (OPENAI_API_KEY not set)")

In [ ]:
# Test 2: cross-table JOIN query
if OPENAI_AVAILABLE:
    print("--- Test: cross-table JOIN query ---")
    result_join, _ = agent.run(
        "Show the total revenue per Product_Class_Code for 2023, ordered from highest to lowest.",
        no_vis=True,
    )
    sql = result_join.get('sql_query', '').strip()
    print(f"SQL: {sql}")
    print(f"\nData preview (first 5 rows):\n{result_join.get('data', '')[:400]}")
    print(f"\nAnswer: {result_join.get('answer', [''])[0][:200]}")

    # The SQL should reference both tables
    sql_upper = sql.upper()
    assert "SALES" in sql_upper, "SQL should reference the sales table"
    assert "PRODUCTS" in sql_upper, "SQL should reference the products table for Product_Class_Code"
    assert "JOIN" in sql_upper, "SQL should contain a JOIN to combine the two tables"
    assert "error" not in result_join or not result_join["error"]
    print("\nCross-table JOIN query: PASSED")
else:
    print("Skipping (OPENAI_API_KEY not set)")

In [ ]:
# Test 3: full pipeline with visualization
if OPENAI_AVAILABLE:
    print("--- Test: full pipeline (lookup + analyze + visualize) ---")
    result_full, _ = agent.run(
        "Show a bar chart of total revenue by Product_Class_Code for 2022.",
        visualization_goal="Bar chart comparing revenue across product classes",
    )
    print(f"SQL: {result_full.get('sql_query', '').strip()}")
    print(f"Chart config: {result_full.get('chart_config')}")
    print(f"Answer steps: {len(result_full.get('answer', []))}")
    assert result_full.get('chart_config') is not None, "chart_config should be set"
    assert len(result_full.get('answer', [])) >= 2, "Should have at least analysis + chart code"
    print("\nFull pipeline multi-table test: PASSED")
else:
    print("Skipping (OPENAI_API_KEY not set)")

In [ ]:
# Render the chart from the full pipeline test (if available)
if OPENAI_AVAILABLE and 'result_full' in dir():
    chart_code = result_full.get("answer", [])[-1]
    data_df = result_full.get("data_df")
    config = result_full.get("chart_config")
    if chart_code and data_df is not None:
        import matplotlib.pyplot as plt
        import pandas as pd
        exec(chart_code, {"data_df": data_df, "config": config, "plt": plt, "pd": pd})
    else:
        print("No chart code or data available to render")
else:
    print("Skipping chart render (OPENAI_API_KEY not set)")

## 7. Table-Selection Path (compact_threshold forced low)

**Purpose:** Force the two-step table-selection approach by setting `compact_threshold=1`. With 2 tables and threshold=1, `should_use_table_selection()` returns True, so the agent first asks the LLM which tables are needed (using only names + descriptions), then passes full column details for only those tables to SQL generation. This is the scalability path for 10+ table schemas.

In [ ]:
# Build schema with low threshold to force table selection
schema_with_selection = DatabaseSchema(
    tables=multitable_schema.tables,
    compact_threshold=1,  # 2 tables > 1 → triggers two-step table selection
)

print(f"Tables: {[t.name for t in schema_with_selection.tables]}")
print(f"compact_threshold: {schema_with_selection.compact_threshold}")
print(f"should_use_table_selection: {schema_with_selection.should_use_table_selection()}")
assert schema_with_selection.should_use_table_selection() is True

print("\n=== Compact summary (what LLM sees in table-selection step) ===")
print(schema_with_selection.get_compact_summary())

In [ ]:
agent_with_selection = None
if OPENAI_AVAILABLE:
    agent_with_selection = SalesDataWorkflow(
        model="gpt-4o-mini",
        schema=schema_with_selection,
    )

    print("--- Test: table-selection path, single-table question ---")
    print("(LLM should select only 'sales', not 'products')")
    result_sel_single, _ = agent_with_selection.run_script(
        "How many units were sold on promotion in 2023?",
        no_vis=True,
    )
    print(f"SQL: {result_sel_single.get('sql_query', '').strip()}")
    print(f"Answer: {result_sel_single.get('answer', [''])[0][:150]}")
    assert "error" not in result_sel_single or not result_sel_single["error"]
    print("\nTable-selection path (single-table Q): PASSED")
else:
    print("Skipping (OPENAI_API_KEY not set)")

In [ ]:
if OPENAI_AVAILABLE:
    print("--- Test: table-selection path, cross-table question ---")
    print("(LLM should select both 'sales' and 'products')")
    result_sel_join, _ = agent_with_selection.run(
        "Which Product_Class_Code had the highest total revenue in 2022?",
        no_vis=True,
    )
    sql = result_sel_join.get('sql_query', '').strip()
    print(f"SQL: {sql}")
    print(f"Answer: {result_sel_join.get('answer', [''])[0][:200]}")

    sql_upper = sql.upper()
    assert "SALES" in sql_upper
    assert "PRODUCTS" in sql_upper
    assert "error" not in result_sel_join or not result_sel_join["error"]
    print("\nTable-selection path (cross-table Q): PASSED")
else:
    print("Skipping (OPENAI_API_KEY not set)")

## 8. API Schema Dict Parsing

**Purpose:** Verify that `_schema_from_dict()` correctly parses the JSON payload format accepted by the `/call-agent` API endpoint. This is the only way to pass a multi-table schema to the agent through the REST API.

In [ ]:
api_payload_schema = {
    "compact_threshold": 3,
    "tables": [
        {
            "name": "sales",
            "description": "Daily sales transactions",
            "file_path": SALES_PATH,
            "columns": [
                {"name": "Store_Number", "description": "Store identifier", "data_type": "INTEGER"},
                {"name": "SKU_Coded", "description": "Product SKU", "data_type": "INTEGER"},
                {"name": "Sold_Date", "description": "Transaction date", "data_type": "DATE",
                 "nullable": False},
                {"name": "Total_Sale_Value", "description": "Revenue in USD", "data_type": "FLOAT",
                 "example_values": ["10.5", "99.0", "250.0"]},
            ]
        },
        {
            "name": "products",
            "description": "Product catalog",
            "file_path": PRODUCTS_PATH,
            "columns": [
                {"name": "SKU_Coded", "description": "Product SKU (PK)", "data_type": "INTEGER"},
                {"name": "Product_Class_Code", "description": "Product class", "data_type": "INTEGER"},
            ]
        }
    ]
}

parsed_schema = _schema_from_dict(api_payload_schema)

print(f"Parsed schema type: {type(parsed_schema).__name__}")
print(f"Tables: {[t.name for t in parsed_schema.tables]}")
print(f"compact_threshold: {parsed_schema.compact_threshold}")

sales = parsed_schema.get_table("sales")
assert sales is not None
assert len(sales.columns) == 4
assert sales.columns[2].nullable is False       # Sold_Date
assert sales.columns[3].example_values == ["10.5", "99.0", "250.0"]  # Total_Sale_Value
assert parsed_schema.compact_threshold == 3

print("\nFull schema from parsed dict:")
print(parsed_schema.get_full_schema_str())
print("API schema dict parsing: PASSED")

In [ ]:
# Test missing optional fields use defaults
minimal_payload = {
    "tables": [
        {
            "name": "sales",
            "file_path": SALES_PATH,
            # description omitted → defaults to name
            "columns": [
                {"name": "Total_Sale_Value"}  # description and data_type omitted
            ]
        }
    ]
    # compact_threshold omitted → defaults to 5
}

minimal_schema = _schema_from_dict(minimal_payload)
assert minimal_schema.compact_threshold == 5               # default
assert minimal_schema.tables[0].description == "sales"     # name used as fallback
assert minimal_schema.tables[0].columns[0].data_type == "VARCHAR"  # default type
assert minimal_schema.tables[0].columns[0].description == "Total_Sale_Value"  # name as fallback

print("Minimal payload defaults:")
print(f"  compact_threshold: {minimal_schema.compact_threshold}")
print(f"  table description: '{minimal_schema.tables[0].description}'")
print(f"  column data_type: '{minimal_schema.tables[0].columns[0].data_type}'")
print("\nMinimal payload defaults: PASSED")

## 9. Cleanup

In [ ]:
shutil.rmtree(TEST_DATA_DIR)
print(f"Cleaned up test data directory: {TEST_DATA_DIR}")

## 10. Summary

This notebook tested:

1. **Schema classes** — `ColumnSchema`, `TableSchema`, `DatabaseSchema` field validation and helper methods
2. **`get_compact_summary()`** — table names + descriptions only (for table-selection prompt)
3. **`get_full_schema_str()`** — full column details, with optional filtering by table name
4. **`should_use_table_selection()`** — threshold logic for switching to two-step mode
5. **Backward compatibility** — `SalesDataAgent()` with no schema still works as before
6. **Two-table agent** — single-table queries, JOIN queries, and full pipeline with chart
7. **Table-selection path** — two-step (compact → select → full SQL) with `compact_threshold=1`
8. **API parsing** — `_schema_from_dict()` with full payload and minimal defaults